# 1. 기본 텐서 연산을 사용한 다중 선형 회귀 모델 구현


1.1. 데이터 준비
x_train 변수는 집의 주요 특성을 담은 입력 데이터를 나타냅니다. 우리는 '집의 크기(size)'와 '방의 개수(rooms)'라는 두 가지를 월세를 예측하기 위한 집의 특성으로 선택했습니다. x_train에는 두 개의 집에 대한 정보가 담겨 있습니다. 첫 번째 집은 방의 크기가 10평이며 방이 하나 있고, 두 번째 집은 방의 크기가 15평이며 방이 두 개 있습니다.

y_train 변수는 이 집들의 가격을 나타내는 타겟 데이터입니다. 첫 번째 집의 월세 가격은 40 두 번째 집의 월세는 70으로 설정했습니다.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)

# 특성: 방의 크기 (size), 방의 개수 (rooms)
x_train = torch.tensor([[10, 1], 
                        [15, 2]], dtype=torch.float)
# 10평 방1개, 15평 방2개
# 타겟: 주택 가격 (price)
y_train = torch.tensor([[40], 
                        [70]], dtype=torch.float)
#월세 40만원, 70만원
print(x_train)
print(y_train)

tensor([[10.,  1.],
        [15.,  2.]])
tensor([[40.],
        [70.]])


## 1.2.가중치와 편향 초기화  
- 단순 선형 회귀(Y= XW + b)에서는 각 독립 변수에 대해 하나의 가중치를 설정하고, 전체 모델에 하나의 편향을 추가했었어요.
- 다중 선형 회귀(Y= X1W1 + X2W2 + b)에서는 독립 변수가 두 개(집의 크기, 방의 개수) 있으므로, 두 개의 가중치와 하나의 편향을 초기화할 필요가 있습니다.

In [3]:
W = torch.zeros(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)  

print("초기 가중치:", W)
print("초기 편향:", b)

초기 가중치: tensor([0., 0.], requires_grad=True)
초기 편향: tensor([0.], requires_grad=True)


## 1.3.옵티마이저 설정 : 확률적 경사 하강법(Stochastic Gradient Descent, SGD) 



In [5]:
optimizer = optim.SGD([W, b], lr= 0.001)
# 확률적 경사 하강법(Stochastic Gradient Descent, SGD) 옵티마이저를 사용하여 변수들의 최적화를 설정합니다. 
# 최적화할 파라미터는 W와 b 입니다. 학습률은 0.001 입니다.

# 다중 선형 회귀의 예측 연산 

- 다중 선형 회귀에서는 여러 독립 변수가 있기 때문에, 각 입력 특성과 해당 가중치의 곱을 모두 더한 다음 편향을 더해야 한다. 
- 이를 효율적으로 계산하기 위해 행렬 곱셈을 사용 

`W.unsqueeze(1)`은 W의 차원을 (2,)에서 (2,1)로 변경 -> x_train과 W사이의 행렬 곱셈이 가능해짐. 

In [9]:
print(f'W shape : {W.shape}')
print(f'W.unsqueeze(1) shape : {W.unsqueeze(1).shape}')
print(f'x_train shape : {x_train.shape}')


W shape : torch.Size([2])
W.unsqueeze(1) shape : torch.Size([2, 1])
x_train shape : torch.Size([2, 2])


In [ ]:
epochs = 10000

for epoch in range(epochs):

    pred = x_train.mm(W.unsqueeze(1)) + b
    loss = torch.mean((pred - y_train) ** 2)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

torch.Size([2, 1])
Epoch [100/10000], Loss: 0.6762
Epoch [200/10000], Loss: 0.6557
Epoch [300/10000], Loss: 0.6359
Epoch [400/10000], Loss: 0.6166
Epoch [500/10000], Loss: 0.5979
Epoch [600/10000], Loss: 0.5798
Epoch [700/10000], Loss: 0.5623
Epoch [800/10000], Loss: 0.5453
Epoch [900/10000], Loss: 0.5288
Epoch [1000/10000], Loss: 0.5128
Epoch [1100/10000], Loss: 0.4972
Epoch [1200/10000], Loss: 0.4822
Epoch [1300/10000], Loss: 0.4676
Epoch [1400/10000], Loss: 0.4534
Epoch [1500/10000], Loss: 0.4397
Epoch [1600/10000], Loss: 0.4264
Epoch [1700/10000], Loss: 0.4135
Epoch [1800/10000], Loss: 0.4010
Epoch [1900/10000], Loss: 0.3888
Epoch [2000/10000], Loss: 0.3771
Epoch [2100/10000], Loss: 0.3657
Epoch [2200/10000], Loss: 0.3546
Epoch [2300/10000], Loss: 0.3438
Epoch [2400/10000], Loss: 0.3334
Epoch [2500/10000], Loss: 0.3233
Epoch [2600/10000], Loss: 0.3136
Epoch [2700/10000], Loss: 0.3041
Epoch [2800/10000], Loss: 0.2949
Epoch [2900/10000], Loss: 0.2859
Epoch [3000/10000], Loss: 0.2773


# 1.5 모델 예측 



In [10]:
test_x = torch.tensor([[20, 3]], dtype=torch.float)  # 예: 20제곱미터 크기에 방 3개인 집

# 예측 수행
with torch.no_grad():  
    pred_y = test_x.mm(W.unsqueeze(1)) + b
    print(f"새로운 데이터에 대한 예측 가격: {pred_y.item()}")

새로운 데이터에 대한 예측 가격: 99.50724792480469


# 2. nn.Linear를 사용하여 직접 구현 

In [11]:
x_train = torch.tensor([[10, 1], 
                        [15, 2]], dtype=torch.float)

y_train = torch.tensor([[40], 
                        [70]], dtype=torch.float)

# 입력 및 출력 특성의 크기 계산
in_features = x_train.shape[1]
out_features = y_train.shape[1]

# 선형 회귀 모델 초기화
model = nn.Linear(in_features=2, out_features=1)

for param in model.parameters():
    print(param.data)

tensor([[0.5406, 0.5869]])
tensor([-0.1657])


# 2.1 모델 초기화와 파라미터 확인

In [12]:
x_train = torch.tensor([[10, 1], 
                        [15, 2]], dtype=torch.float)

y_train = torch.tensor([[40], 
                        [70]], dtype=torch.float)

# 입력 및 출력 특성의 크기 계산
in_features = x_train.shape[1]
out_features = y_train.shape[1]

# 선형 회귀 모델 초기화
model = nn.Linear(in_features, out_features)

for param in model.parameters():
    print(param.data)

tensor([[ 0.6496, -0.1549]])
tensor([0.1427])


In [13]:
import torch.optim as optim
import torch.nn.functional as F

optimizer = optim.SGD(model.parameters(), lr=0.001)

epochs = 10000
for epoch in range(epochs):
    pred = model(x_train)
    loss = F.mse_loss(pred, y_train)
    optimizer.zero_grad()  
    loss.backward()        
    optimizer.step()       

    if (epoch+1) % 100 == 0:
         print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


test_x = torch.tensor([[20, 3]], dtype=torch.float)  # 예: 120제곱미터 크기에 방 3개인 집

# 예측 수행
with torch.no_grad():  
    pred_y = model(test_x)
    print(f"새로운 데이터에 대한 예측 가격: {pred_y.item()}")

Epoch [100/10000], Loss: 15.1104
Epoch [200/10000], Loss: 14.6530
Epoch [300/10000], Loss: 14.2094
Epoch [400/10000], Loss: 13.7793
Epoch [500/10000], Loss: 13.3621
Epoch [600/10000], Loss: 12.9577
Epoch [700/10000], Loss: 12.5654
Epoch [800/10000], Loss: 12.1850
Epoch [900/10000], Loss: 11.8161
Epoch [1000/10000], Loss: 11.4585
Epoch [1100/10000], Loss: 11.1116
Epoch [1200/10000], Loss: 10.7752
Epoch [1300/10000], Loss: 10.4490
Epoch [1400/10000], Loss: 10.1327
Epoch [1500/10000], Loss: 9.8260
Epoch [1600/10000], Loss: 9.5285
Epoch [1700/10000], Loss: 9.2401
Epoch [1800/10000], Loss: 8.9604
Epoch [1900/10000], Loss: 8.6891
Epoch [2000/10000], Loss: 8.4261
Epoch [2100/10000], Loss: 8.1710
Epoch [2200/10000], Loss: 7.9237
Epoch [2300/10000], Loss: 7.6838
Epoch [2400/10000], Loss: 7.4512
Epoch [2500/10000], Loss: 7.2256
Epoch [2600/10000], Loss: 7.0069
Epoch [2700/10000], Loss: 6.7948
Epoch [2800/10000], Loss: 6.5891
Epoch [2900/10000], Loss: 6.3896
Epoch [3000/10000], Loss: 6.1962
Epoch

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

x_train = torch.tensor([[10, 1], 
                        [15, 2]], dtype=torch.float)

y_train = torch.tensor([[40], 
                        [70]], dtype=torch.float)


# 모델 정의
class LinearRegressionModel(nn.Module):
    def __init__(self, in_features, out_features):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)

# 모델 인스턴스 생성
model = LinearRegressionModel(x_train.shape[1], y_train.shape[1])

In [18]:
optimizer = optim.SGD(model.parameters(), lr=0.001)

nb_epochs = 10000
for epoch in range(nb_epochs):

    pred = model(x_train)
    loss = F.mse_loss(pred, y_train)

    optimizer.zero_grad()  
    loss.backward()        
    optimizer.step()       

    if (epoch+1) % 100 == 0:
         print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [100/10000], Loss: 13.8919
Epoch [200/10000], Loss: 13.4714
Epoch [300/10000], Loss: 13.0636
Epoch [400/10000], Loss: 12.6681
Epoch [500/10000], Loss: 12.2846
Epoch [600/10000], Loss: 11.9128
Epoch [700/10000], Loss: 11.5521
Epoch [800/10000], Loss: 11.2024
Epoch [900/10000], Loss: 10.8633
Epoch [1000/10000], Loss: 10.5344
Epoch [1100/10000], Loss: 10.2155
Epoch [1200/10000], Loss: 9.9063
Epoch [1300/10000], Loss: 9.6064
Epoch [1400/10000], Loss: 9.3156
Epoch [1500/10000], Loss: 9.0336
Epoch [1600/10000], Loss: 8.7602
Epoch [1700/10000], Loss: 8.4950
Epoch [1800/10000], Loss: 8.2378
Epoch [1900/10000], Loss: 7.9884
Epoch [2000/10000], Loss: 7.7466
Epoch [2100/10000], Loss: 7.5121
Epoch [2200/10000], Loss: 7.2847
Epoch [2300/10000], Loss: 7.0642
Epoch [2400/10000], Loss: 6.8503
Epoch [2500/10000], Loss: 6.6430
Epoch [2600/10000], Loss: 6.4419
Epoch [2700/10000], Loss: 6.2469
Epoch [2800/10000], Loss: 6.0578
Epoch [2900/10000], Loss: 5.8744
Epoch [3000/10000], Loss: 5.6966
Epoch [3

In [19]:
new_x = torch.tensor([[12, 3]], dtype=torch.float)  

with torch.no_grad(): 
    predicted_price = model(new_x)
    print(f"새로운 데이터에 대한 예측 가격: {predicted_price.item()} 만 달러")

새로운 데이터에 대한 예측 가격: 66.30992889404297 만 달러
